# Trabajo Práctico: RAG sobre Legislación Argentina

**Materia:** Aplicaciones con Modelos de Lenguaje (LLMs)
**Opción elegida:** A — RAG (Retrieval-Augmented Generation)
**Alumna:** *(completar)*

---

## Objetivo del sistema

Construir un asistente que responda preguntas sobre un corpus de **legislación argentina** (leyes y normas publicadas en fuentes oficiales como InfoLEG y el Boletín Oficial), usando **RAG**: en lugar de que el modelo responda "de memoria", primero **busca** la información en los documentos y luego **redacta** la respuesta en base a lo que encontró.

Al final comparamos las respuestas **CON RAG** vs **SIN RAG** para 5 preguntas.


## ¿Qué es RAG?

**Qué es:** RAG (*Retrieval-Augmented Generation*, "generación aumentada por recuperación") es una técnica que combina dos pasos: primero **recupera** fragmentos relevantes de una base de documentos y después se los pasa al modelo de lenguaje para que **genere** la respuesta usando esa información.

**Para qué sirve:** para que el modelo responda sobre información que no tiene aprendida (nuestros documentos privados o muy específicos) y para **reducir las alucinaciones** (respuestas inventadas), porque lo obligamos a basarse en un texto concreto.

**Analogía:** es la diferencia entre rendir un examen **de memoria** y rendirlo **a libro abierto**. Sin RAG, el modelo contesta con lo que recuerda (y a veces inventa). Con RAG, primero abre el "libro" (nuestros documentos), busca la parte que corresponde y recién ahí responde.

### El pipeline que vamos a construir

```
PDFs  ->  Chunking  ->  Embeddings  ->  Base vectorial (Chroma)
                                                  |
                       Pregunta  ->  busqueda top-k  ->  Contexto + Pregunta  ->  LLM  ->  Respuesta
```

| Paso | Que hace |
|------|----------|
| **Chunking** | Corta los documentos largos en fragmentos ("chunks") manejables. |
| **Embeddings** | Convierte cada chunk en un vector de numeros que representa su significado. |
| **Base vectorial** | Guarda esos vectores y permite buscar por similitud. |
| **Top-k** | Ante una pregunta, recupera los *k* fragmentos mas parecidos. |
| **Generacion** | El LLM redacta la respuesta usando esos fragmentos como contexto. |


## 1. Instalacion de dependencias

Instalamos las librerias del pipeline:
- **chromadb**: base de datos vectorial.
- **sentence-transformers**: modelo de embeddings (multilingue, anda bien en espanol).
- **pypdf**: para leer el texto de los PDFs.
- **google-genai**: cliente oficial de Gemini (el LLM que genera las respuestas).
- **pandas**: para mostrar la comparacion en una tabla.

In [ ]:
!pip install -q chromadb sentence-transformers pypdf google-genai pandas
print("Dependencias instaladas.")

## 2. Configuracion de la API de Gemini

Necesitas una **API key gratuita** de Gemini. La obtenes en https://aistudio.google.com/apikey (tiene un nivel gratuito suficiente para este TP).

Al ejecutar la celda te va a pedir que la pegues (no queda escrita en el notebook).

In [ ]:
import os
from getpass import getpass

os.environ["GEMINI_API_KEY"] = getpass("Ingresa tu GEMINI_API_KEY: ")

In [ ]:
from google import genai

# Creamos el cliente de Gemini y elegimos el modelo
client = genai.Client(api_key=os.environ["GEMINI_API_KEY"])
MODELO_LLM = "gemini-2.5-flash"

# Prueba rapida para confirmar que la API responde
resp = client.models.generate_content(model=MODELO_LLM, contents="Responde solo con la palabra: OK")
print("Respuesta de Gemini:", resp.text)

## 3. Carga del corpus (minimo 5 documentos)

Pone tus PDFs de legislacion (al menos 5, descargados de InfoLEG o el Boletin Oficial) dentro de una carpeta llamada `documentos`.

- **En Google Colab:** ejecuta la celda de subida (descomentada) para subir los archivos.
- **En tu compu / repositorio Git:** crea la carpeta `documentos/` y copia ahi los PDFs.

In [ ]:
import os
os.makedirs("documentos", exist_ok=True)

# --- Subir PDFs en Google Colab (descomenta si estas en Colab) ---
# from google.colab import files
# subidos = files.upload()
# for nombre in subidos:
#     os.replace(nombre, os.path.join("documentos", nombre))

print("Archivos en la carpeta 'documentos':")
print(os.listdir("documentos"))

In [ ]:
from pypdf import PdfReader

def leer_pdfs(carpeta):
    """Lee todos los PDFs de una carpeta y devuelve su texto + nombre de archivo."""
    documentos = []
    for nombre in sorted(os.listdir(carpeta)):
        if nombre.lower().endswith(".pdf"):
            ruta = os.path.join(carpeta, nombre)
            lector = PdfReader(ruta)
            texto = ""
            for pagina in lector.pages:
                texto += (pagina.extract_text() or "") + "\n"
            documentos.append({"fuente": nombre, "texto": texto})
            print(f"Leido: {nombre}  ({len(texto)} caracteres)")
    return documentos

documentos = leer_pdfs("documentos")
print(f"\nTotal de documentos cargados: {len(documentos)}")
assert len(documentos) >= 5, "El TP pide un minimo de 5 documentos."

## 4. Chunking (division en fragmentos)

**Que es:** cortar cada documento largo en fragmentos mas chicos.

**Para que sirve:** un modelo no procesa bien un texto enorme de una sola vez, y ademas la busqueda es mas precisa si los pedazos son chicos. El **solapamiento** (overlap) hace que dos chunks consecutivos compartan un poco de texto, para no cortar una idea justo a la mitad.

**Analogia:** en vez de resaltar un libro entero, lo dividis en parrafos: despues es mucho mas facil encontrar el parrafo exacto que responde tu pregunta.

In [ ]:
def dividir_en_chunks(texto, tam=1000, solapamiento=150):
    """Divide un texto en fragmentos de 'tam' caracteres con solapamiento."""
    chunks = []
    inicio = 0
    while inicio < len(texto):
        fragmento = texto[inicio:inicio + tam].strip()
        if fragmento:
            chunks.append(fragmento)
        inicio += tam - solapamiento
    return chunks

# Aplicamos el chunking a todos los documentos,
# guardando de que archivo (fuente) proviene cada chunk -> metadatos
todos_chunks, metadatos, ids = [], [], []
contador = 0
for doc in documentos:
    for chunk in dividir_en_chunks(doc["texto"]):
        todos_chunks.append(chunk)
        metadatos.append({"fuente": doc["fuente"]})
        ids.append(f"chunk_{contador}")
        contador += 1

print(f"Total de chunks generados: {len(todos_chunks)}")
print("\nEjemplo de chunk:\n", todos_chunks[0][:300], "...")

## 5. Embeddings y base vectorial (Chroma)

**Que son los embeddings:** convertir texto en un vector de numeros que captura su **significado**. Dos textos que hablan de lo mismo quedan "cerca" en ese espacio numerico, aunque usen palabras distintas.

**Para que sirve la base vectorial:** guarda todos esos vectores y permite buscar, ante una pregunta, cuales son los fragmentos mas parecidos **por significado** (no por coincidencia exacta de palabras).

**Analogia:** es como un mapa donde los textos que tratan temas parecidos quedan agrupados en el mismo barrio. Buscar deja de ser "encontrar la palabra igual" y pasa a ser "ir al barrio correcto".

Usamos el modelo multilingue `paraphrase-multilingual-MiniLM-L12-v2`, que funciona bien en espanol. Chroma calcula los embeddings automaticamente al cargar los chunks.

In [ ]:
import chromadb
from chromadb.utils import embedding_functions

# Modelo de embeddings multilingue (la primera vez se descarga, tarda un poco)
funcion_embeddings = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name="paraphrase-multilingual-MiniLM-L12-v2"
)

cliente_chroma = chromadb.Client()

# Si ya existe la coleccion (por re-ejecutar la celda), la borramos para empezar limpio
try:
    cliente_chroma.delete_collection("legislacion")
except Exception:
    pass

coleccion = cliente_chroma.create_collection(
    name="legislacion",
    embedding_function=funcion_embeddings,
)

# Cargamos los chunks: Chroma genera los embeddings y guarda los metadatos (la fuente)
coleccion.add(documents=todos_chunks, metadatas=metadatos, ids=ids)
print(f"Chunks cargados en la base vectorial: {coleccion.count()}")

## 6. Recuperacion (busqueda top-k)

**Que es:** dada una pregunta, buscar en la base vectorial los *k* fragmentos mas parecidos.

**Para que sirve:** son esos fragmentos los que despues le vamos a pasar al LLM como "contexto" para que responda con informacion real de los documentos.

In [ ]:
def recuperar(pregunta, k=4):
    """Devuelve los k chunks mas parecidos a la pregunta, con su fuente."""
    resultado = coleccion.query(query_texts=[pregunta], n_results=k)
    chunks = resultado["documents"][0]
    fuentes = [m["fuente"] for m in resultado["metadatas"][0]]
    return chunks, fuentes

# Prueba: cambia la pregunta por algo que este en tus documentos
chunks, fuentes = recuperar("Cual es el objeto de la ley?", k=3)
for i, (c, f) in enumerate(zip(chunks, fuentes), start=1):
    print(f"[{i}] Fuente: {f}\n{c[:250]}...\n")

## 7. Generacion SIN RAG (linea de base)

Le preguntamos directamente al modelo, **sin darle los documentos**. Esto es nuestra base de comparacion: aca el modelo responde solo con lo que "recuerda" de su entrenamiento.

In [ ]:
def responder_sin_rag(pregunta):
    """Le pregunta directamente al LLM, sin contexto de los documentos."""
    resp = client.models.generate_content(model=MODELO_LLM, contents=pregunta)
    return resp.text

## 8. Generacion CON RAG

Recuperamos los fragmentos relevantes y armamos un **prompt** que le da al modelo el contexto + la pregunta, con la instruccion de responder **solo** con esa informacion. Si no esta, debe decir que no tiene datos suficientes (asi evitamos que invente).

In [ ]:
def responder_con_rag(pregunta, k=4):
    """Recupera contexto de los documentos y se lo pasa al LLM."""
    chunks, fuentes = recuperar(pregunta, k=k)
    contexto = "\n\n".join(chunks)
    prompt = f"""Sos un asistente que responde SOLO con el contexto dado.
Si la respuesta no esta en el contexto, responde: "No tengo informacion suficiente en los documentos".

CONTEXTO:
{contexto}

PREGUNTA: {pregunta}

RESPUESTA:"""
    resp = client.models.generate_content(model=MODELO_LLM, contents=prompt)
    return resp.text, fuentes

## 9. Evaluacion: CON RAG vs SIN RAG

Comparamos las respuestas para 5 preguntas. Lo esperable es que **con RAG** las respuestas sean mas precisas, se ajusten a *tus* documentos e indiquen la fuente, mientras que **sin RAG** el modelo puede ser vago o inventar.

> **Importante:** reemplaza las preguntas por otras que se puedan responder con TUS documentos concretos.

In [ ]:
import pandas as pd

preguntas = [
    "Cual es el objeto o finalidad de la ley?",
    "A quienes se aplica la norma?",
    "Que obligaciones principales establece?",
    "Que sanciones o penalidades preve?",
    "Cual es la autoridad de aplicacion?",
]

filas = []
for p in preguntas:
    print(f"Procesando: {p}")
    sin_rag = responder_sin_rag(p)
    con_rag, fuentes = responder_con_rag(p)
    filas.append({
        "Pregunta": p,
        "Respuesta SIN RAG": sin_rag,
        "Respuesta CON RAG": con_rag,
        "Fuentes recuperadas": ", ".join(sorted(set(fuentes))),
    })

df = pd.DataFrame(filas)
pd.set_option("display.max_colwidth", None)
df

### Ver una comparacion en detalle (opcional)

Ejecuta esta celda para leer una pregunta puntual de forma mas comoda.

In [ ]:
i = 0  # cambia el indice (0 a 4) para ver otra pregunta

print("PREGUNTA:", filas[i]["Pregunta"])
print("\n--- SIN RAG ---\n", filas[i]["Respuesta SIN RAG"])
print("\n--- CON RAG ---\n", filas[i]["Respuesta CON RAG"])
print("\nFuentes:", filas[i]["Fuentes recuperadas"])

## 10. Conclusiones y analisis critico

**Que observamos:** *(completa con lo que veas en tus resultados)* Con RAG las respuestas tienden a ser mas precisas y ancladas a los documentos, e indican de que norma salio la informacion. Sin RAG, el modelo responde de forma mas generica y puede equivocarse en datos especificos (numeros de articulos, fechas, autoridades).

**Limitaciones:**
- La calidad depende del **chunking**: si los fragmentos son muy grandes o muy chicos, la busqueda pierde precision.
- Si el PDF esta escaneado (imagen), `pypdf` no extrae bien el texto; haria falta OCR.
- El sistema solo sabe lo que esta en el corpus: si falta una ley, no puede responder sobre ella.
- La recuperacion por similitud puede traer fragmentos parecidos pero no exactos a lo que se pregunta.

**Posibles mejoras:**
- Probar distintos tamanos de chunk y valores de *k*.
- Agregar **citas exactas** (articulo y fuente) en cada respuesta.
- Usar un modelo de embeddings mas potente o un *re-ranker*.
- Ampliar el corpus con mas normativa y mantenerlo actualizado desde InfoLEG / Boletin Oficial.
